# Projet Recherche Opérationnelle : Livrable Check
## Modélisation du problème VRPTW

---

Établissement : CESI : PGE A3 FISA INFO  
Contexte : Réponse à l'appel à manifestation d'intérêt de l'ADEME  
Auteurs : Fayçal Rguig, Rayene Medjtoh, Yanis Bendehane, Victor Schentuleit
Date : 02/04/2026  
Version : 0.1 : Livrable check

---

### Objectif de ce notebook

Ce notebook constitue le livrable check du projet de Recherche Opérationnelle.  
Il présente :

1. La modélisation formelle du problème de tournées de véhicules avec fenêtres temporelles (VRPTW)
2. L'analyse de sa complexité théorique et la démonstration de sa NP-difficulté
3. Le générateur d'instances aléatoires réutilisé par toutes les phases suivantes

Les méthodes de résolution (heuristiques, métaheuristiques, Deep Learning) sont traitées dans le livrable final.

In [ ]:
# ── Imports Globaux ──
import numpy as np #Matrice & calcul numérique
import matplotlib.pyplot as plt #Visualisation
import matplotlib.patches as mpatches #Pour les légendes personnalisées
import networkx as nx #Graphes et algorithmes de graphes
import json #Pour la lecture de fichiers JSON
import time #Pour mesurer le temps d'exécution
import math #Pour les fonctions mathématiques
from itertools import permutations #Pour générer des permutations de listes

# ── Configuration affichage ────S'appliquera a tout les plots──
plt.rcParams['figure.figsize'] = (10, 6) #runtime confguration parameters
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── Seed globale pour reproductibilité multi methodes ───
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print("Environnement chargé.")
print(f"numpy  {np.__version__}")
print(f"networkx  {nx.__version__}")

---

## 1. Contexte et motivation

### 1.1 L'appel à manifestation d'intérêt de l'ADEME

Depuis les années 90, la réduction des émissions de gaz à effet de serre est devenue 
un enjeu mondial majeur. Le protocole de Kyoto (1997), puis les engagements plus 
ambitieux qui ont suivi -- comme la division par 4 des émissions françaises d'ici 2050 
-- ont placé la question de la mobilité au cœur des priorités environnementales.

L'ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie) a récemment 
lancé un appel à manifestation d'intérêt pour promouvoir des solutions de mobilité 
intelligente adaptées aux territoires. Les applications visées sont nombreuses :

- Distribution du courrier et livraison de colis
- Collecte et traitement des déchets
- Maintenance des équipements urbains (éclairage public, signalisation)
- Transport de personnes en zones peu denses

Notre structure CesiCDP répond à cet appel. L'enjeu est double : proposer 
une solution algorithmique robuste, et démontrer son impact environnemental concret 
par la réduction des kilomètres parcourus et de la consommation de carburant.

### 1.2 Du problème concret au problème algorithmique

Le problème opérationnel est le suivant :

> Chaque jour, des véhicules doivent partir d'un dépôt, livrer un ensemble de 
> clients, puis retourner au dépôt. Chaque client doit être visité exactement une 
> fois, dans un créneau horaire défini, sans dépasser la capacité du véhicule. 
> L'objectif est de minimiser la durée totale des tournées.

Ce problème est connu dans la littérature sous le nom de VRPTW 
(*Vehicle Routing Problem with Time Windows*). Il s'agit d'une extension enrichie 
du célèbre TSP (*Travelling Salesman Problem*), lui-même l'un des problèmes 
les plus étudiés en informatique et en mathématiques depuis les années 1930.

Les deux contraintes que nous retenons pour cette étude sont :

| Contrainte | Description | Justification |
|---|---|---|
| Fenêtres temporelles | Chaque client ne peut être visité qu'entre $a_i$ et $b_i$ | Réalité métier : créneaux de livraison imposés |
| Multi-véhicules + capacité | Plusieurs véhicules disponibles, charge maximale $Q$ | Réalité logistique : flotte limitée en tonnage |

Ces deux contraintes combinées forment le VRPTW, problème de référence 
en recherche opérationnelle, pour lequel des benchmarks standardisés existent 
(instances Solomon, 1987).

### 1.3 Pourquoi modéliser formellement ?

Avant d'écrire le moindre algorithme, il est indispensable de traduire ce problème 
réel en objets mathématiques précis. Cette étape de modélisation remplit trois rôles :

1. Clarifier : une définition formelle évite les ambiguïtés. 
   "Minimiser la durée" peut vouloir dire minimiser la distance totale, 
   le nombre de véhicules, ou le retard cumulé. La modélisation force à choisir.

2. Prouver : la représentation formelle permet de démontrer des propriétés 
   théoriques, notamment la complexité du problème, ce qui justifie les choix 
   algorithmiques qui suivront.

3. Implémenter : un algorithme ne manipule pas des "villes" et des "routes" 
   abstraites, mais des matrices, des vecteurs, des indices. 
   La modélisation est le pont entre la réalité et le code.

Dans la suite de ce notebook, nous construisons pas à pas le modèle mathématique 
complet du VRPTW, puis nous analysons sa complexité théorique, 
et enfin nous implémentons un générateur d'instances pour alimenter 
les phases de résolution suivantes.

---

## 2. Modélisation formelle

### 2.1 Représentation par un graphe

Le réseau routier se modélise naturellement comme un graphe orienté pondéré,
noté $G = (V, A, c)$, où chaque composant traduit un élément du problème réel.

L'ensemble des sommets $V$ représente les lieux à visiter :

$$V = \{0, 1, 2, \ldots, n\}$$

Le sommet $0$ est le dépôt : point de départ et de retour de tous les véhicules.
Les sommets $1$ à $n$ sont les $n$ clients à livrer.

L'ensemble des arcs $A$ représente les trajets possibles entre les lieux :

$$A \subseteq V \times V = \{(i, j) \mid i \in V,\ j \in V,\ i \neq j\}$$

Nous travaillons sur un graphe complet : tout client est directement accessible
depuis n'importe quel autre sommet. Cette hypothèse est réaliste dans un réseau
routier urbain où le chemin le plus court entre deux points peut toujours être calculé.

La fonction de coût $c$ associe à chaque arc $(i, j)$ une valeur réelle positive
représentant la durée de trajet :

$$c : A \rightarrow \mathbb{R}^+, \quad c_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

où $(x_i, y_i)$ sont les coordonnées euclidiennes de la ville $i$.
Cette distance satisfait l'inégalité triangulaire :
$c_{ik} \leq c_{ij} + c_{jk}$ pour tout $i, j, k \in V$,
ce qui garantit qu'un trajet direct est toujours au moins aussi court
qu'un trajet avec étape intermédiaire.

In [ ]:
# Illustration : graphe complet sur 5 sommets (1 dépôt + 4 clients)
np.random.seed(GLOBAL_SEED)

n_example = 4  # nombre de clients (hors dépôt)
coords_example = np.random.rand(n_example + 1, 2) * 100

# Construction du graphe complet orienté
G_example = nx.DiGraph()
for i in range(n_example + 1): # Positions 0 à n_example (0 = dépôt, 1..n_example = clients)
    G_example.add_node(i) #Création des sommets

for i in range(n_example + 1): #Création des arcs avec poids (distance euclidienne)
    for j in range(n_example + 1):
        if i != j:
            dist = np.linalg.norm(coords_example[i] - coords_example[j])
            G_example.add_edge(i, j, weight=round(dist, 1)) #Poids = distance arrondie à 1 décimale

# Affichage
pos = {i: coords_example[i] for i in range(n_example + 1)}
node_colors = ["#CA3807" if i == 0 else "#85AFDA" for i in range(n_example + 1)]
node_labels = {0: 'Dépôt'} | {i: f'Client {i}' for i in range(1, n_example + 1)}

fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx_nodes(G_example, pos, node_color=node_colors,
                       node_size=800, ax=ax)
nx.draw_networkx_labels(G_example, pos, labels=node_labels,
                        font_size=9, font_color='black', ax=ax)
nx.draw_networkx_edges(G_example, pos, alpha=0.3, arrows=True,
                       arrowsize=12, ax=ax,
                       connectionstyle='arc3,rad=0.1')

ax.set_title('Graphe complet G = (V, A, c) : dépôt + 4 clients', pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Sommets : {list(G_example.nodes)}")
print(f"Nombre d'arcs : {G_example.number_of_edges()} (soit n×(n+1) = {n_example}×{n_example+1})")

### 2.2 La matrice des distances

En pratique, les coûts $c_{ij}$ sont stockés sous forme d'une matrice carrée
$C \in \mathbb{R}^{(n+1) \times (n+1)}$ :

$$C = \begin{pmatrix} 0 & c_{01} & c_{02} & \cdots & c_{0n} 
\\ c_{10} & 0 & c_{12} & \cdots & c_{1n} 
\\ \vdots & & \ddots & & \vdots 
\\ \vdots & & & \ddots 
\\ c_{n0} & c_{n1} & \cdots & & 0 \end{pmatrix}$$

La diagonale est nulle ($c_{ii} = 0$, un sommet est à distance 0 de lui-même).
La matrice est symétrique dans le cas euclidien ($c_{ij} = c_{ji}$),
mais cette propriété n'est pas requise par le modèle : on pourrait
avoir des durées différentes selon le sens de circulation.

L'accès à n'importe quelle distance est en temps constant $O(1)$,
ce qui est essentiel pour les algorithmes qui consultent cette matrice
des millions de fois.

In [ ]:
# Calcul vectorisé de la matrice de distances euclidiennes
# coords_example : tableau (n+1, 2) des coordonnées

def compute_distance_matrix(coords):
    """
    Calcule la matrice de distances euclidiennes entre tous les sommets.
    @param coords : np.ndarray de forme (n+1, 2)
    @return : np.ndarray de forme (n+1, n+1) avec les distances
    """
    # Différence entre toutes les paires de coordonnées en une seule opération
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :] # Broadcast simplifier les boucles
    dist = np.sqrt((diff ** 2).sum(axis=2)) # (n+1, n+1) avec les distances euclidiennes
    return dist 

dist_example = compute_distance_matrix(coords_example)

# Affichage formaté
print("Matrice des distances C (arrondie à 1 décimale) :\n")
header = "      " + "  ".join(
    [f"{'Dép':>6}"] + [f"{'C'+str(i):>6}" for i in range(1, n_example + 1)]
)
print(header)
print("  " + "-" * (len(header) - 2))
row_labels = ['Dép'] + [f'C{i}' for i in range(1, n_example + 1)]
for i, label in enumerate(row_labels):
    row = "  ".join([f"{dist_example[i][j]:6.1f}" for j in range(n_example + 1)])
    print(f"{label:>4} | {row}")

print(f"\nDiagonale nulle : {np.all(np.diag(dist_example) == 0)}")
print(f"Matrice symétrique : {np.allclose(dist_example, dist_example.T)}")

### 2.3 Variables de décision

La modélisation du VRPTW relève de la Programmation Linéaire en Nombres Entiers
(PLNE). Elle repose sur deux types de variables de décision : c'est-à-dire
les quantités que le modèle doit déterminer.

Soit $K = \{1, \ldots, m\}$ l'ensemble des $m$ véhicules disponibles.

La première variable encode les décisions de routage :

$$x_{ij}^k \in \{0, 1\} \quad \forall i, j \in V,\ \forall k \in K$$

$x_{ij}^k = 1$ si et seulement si le véhicule $k$ emprunte l'arc $(i \to j)$
dans sa tournée, $0$ sinon. L'ensemble de toutes ces variables binaires
décrit entièrement quelles routes sont empruntées et par quel véhicule.

La seconde variable encode les décisions temporelles :

$$t_i^k \in \mathbb{R}^+ \quad \forall i \in V,\ \forall k \in K$$

$t_i^k$ représente l'heure d'arrivée du véhicule $k$ au client $i$.
Cette variable est indispensable pour modéliser les fenêtres temporelles.
Elle couple les décisions de routage au planning horaire réel :
l'ordre dans lequel un véhicule visite les clients détermine ses heures d'arrivée.

Le nombre total de variables binaires est $|V|^2 \times |K| = (n+1)^2 \times m$,
ce qui croit rapidement avec $n$ : c'est l'une des sources de la difficulté
du problème.

### 2.4 Fonction objectif

L'objectif est de minimiser la somme des coûts de tous les arcs empruntés
par tous les véhicules :

$$\min \sum_{k \in K} \sum_{i \in V} \sum_{j \in V} c_{ij} \cdot x_{ij}^k$$

pour chaque véhicule k, pour chaque ville de départ i, pour chaque ville d'arrivée j : si le véhicule k emprunte l'arc i→j, ajouter le coût de ce trajet. Sommer tout ça.

Le produit $c_{ij} \cdot x_{ij}^k$ vaut $c_{ij}$ si le véhicule $k$ emprunte
l'arc $(i, j)$, et $0$ sinon. La somme triple ne compte donc que les arcs
réellement utilisés dans la solution finale.

Cette formulation minimise la distance totale parcourue par la flotte.
D'autres objectifs sont possibles : minimiser le nombre de véhicules utilisés,
ou minimiser le retard total : mais ils nécessiteraient une reformulation
du modèle. Nous retenons la distance totale pour sa cohérence avec
l'objectif environnemental de l'ADEME.

---

## 3. Contraintes du VRPTW

### 3.1 Du TSP au VRPTW

Le TSP dans sa forme de base cherche une tournée minimale sur un graphe complet.
Notre problème réel impose deux couches de contraintes supplémentaires,
qui le transforment en VRPTW (Vehicle Routing Problem with Time Windows).

La première couche concerne la logistique de la flotte :
chaque véhicule a une capacité maximale $Q$, et plusieurs véhicules
peuvent opérer en parallèle depuis le dépôt.

La seconde couche concerne le temps :
chaque client $i$ ne peut être visité qu'entre une heure d'ouverture $a_i$
et une heure de fermeture $b_i$.

Ces deux contraintes sont justifiées par le contexte ADEME :
les tournées de livraison en milieu urbain sont soumises à des créneaux
imposés par les clients, et les véhicules ont une charge physique limitée.
Leur combinaison produit le VRPTW, problème de référence en recherche
opérationnelle pour lequel des benchmarks standardisés existent
(instances Solomon, 1987).

Pour chaque contrainte, nous présentons la formulation formelle,
une explication en langage naturel, et un exemple numérique concret.

### 3.2 Contrainte C1 : Couverture

$$\sum_{k \in K} \sum_{j \in V} x_{ij}^k = 1 \qquad \forall i \in V \setminus \{0\}$$

Chaque client $i$ (hors dépôt) doit être visité exactement une fois,
par exactement un véhicule. La somme de tous les arcs sortant de $i$
sur tous les véhicules vaut 1.

Cette contrainte interdit deux situations indésirables :
une visite oubliée (somme = 0) et une double visite (somme $\geq 2$).

Exemple numérique avec 3 clients et 2 véhicules :

$$x_{1,2}^1 + x_{1,3}^1 + x_{1,0}^1 + x_{1,2}^2 + x_{1,3}^2 + x_{1,0}^2 = 1$$

Si le véhicule 1 va du client 1 vers le client 3, alors $x_{1,3}^1 = 1$
et tous les autres termes valent 0. La somme vaut bien 1.


### 3.3 Contrainte C2 : Conservation de flux

$$\sum_{i \in V} x_{ij}^k = \sum_{i \in V} x_{ji}^k \qquad \forall j \in V,\ \forall k \in K$$

Pour chaque sommet $j$ et chaque véhicule $k$, le nombre d'arcs entrants
égale le nombre d'arcs sortants. Si un véhicule arrive quelque part,
il doit aussi en repartir.

Cette contrainte garantit que chaque tournée est un cycle fermé.
Elle interdit qu'un véhicule "s'arrête" en cours de route
ou qu'il apparaisse ou disparaisse en un point intermédiaire.

Elle s'applique aussi au dépôt : le véhicule qui part du dépôt (arc sortant)
doit y revenir (arc entrant). C'est la contrainte de retour au dépôt.

Exemple numérique pour le client 2, véhicule 1 :

$$x_{0,2}^1 + x_{1,2}^1 + x_{3,2}^1 = x_{2,0}^1 + x_{2,1}^1 + x_{2,3}^1$$

Si le véhicule 1 arrive au client 2 depuis le client 1
et repart vers le client 3, alors $x_{1,2}^1 = 1$ et $x_{2,3}^1 = 1$,
les autres termes valent 0. L'égalité $1 = 1$ est vérifiée.

### 3.4 Contrainte C3 : Capacité des véhicules

$$\sum_{i \in V} q_i \cdot \left(\sum_{j \in V} x_{ij}^k\right) \leq Q \qquad \forall k \in K$$

La somme des demandes de tous les clients visités par le véhicule $k$
ne peut pas dépasser sa capacité maximale $Q$.

Le terme $\sum_{j \in V} x_{ij}^k$ vaut 1 si le client $i$ est visité
par le véhicule $k$, 0 sinon. Le produit $q_i \cdot (\ldots)$
ne prend donc en compte que les clients effectivement servis.

Lorsque cette contrainte est active, le problème ne peut pas être résolu
par un seul véhicule : il faut répartir les clients entre plusieurs tournées.
C'est ce qui fait du VRPTW un problème fondamentalement différent du TSP.

Exemple numérique avec $Q = 100$ kg :

| Client | Demande $q_i$ | Véhicule 1 |
|--------|--------------|------------|
| 1      | 40 kg        | oui        |
| 2      | 35 kg        | oui        |
| 3      | 50 kg        | non        |

Charge véhicule 1 : $40 + 35 = 75 \leq 100$ : contrainte respectée.
Si on ajoutait le client 3 : $40 + 35 + 50 = 125 > 100$ : infaisable,
il faut un second véhicule pour le client 3.

### 3.5 Contrainte C4 : Fenêtres temporelles

$$a_i \leq t_i^k \leq b_i \qquad \forall i \in V,\ \forall k \in K$$

L'heure d'arrivée $t_i^k$ du véhicule $k$ chez le client $i$
doit se situer dans l'intervalle $[a_i, b_i]$.

Trois situations se présentent selon l'heure d'arrivée effective :

- Si $t_i^k \in [a_i, b_i]$ : la visite est valide, le service commence immédiatement.
- Si $t_i^k < a_i$ : le véhicule arrive trop tôt et attend.
  L'heure de début de service devient $a_i$.
  Ce temps d'attente est autorisé mais s'accumule et retarde les visites suivantes.
- Si $t_i^k > b_i$ : la fenêtre est fermée. En modélisation stricte (hard constraint),
  cette solution est infaisable. En modélisation souple (soft constraint),
  une pénalité proportionnelle au retard est ajoutée à la fonction objectif.

Nous retenons la modélisation souple pour les phases algorithmiques,
ce qui permet aux heuristiques et au recuit simulé d'explorer
des solutions temporairement infaisables :

$$\text{pénalité}_i^k = \lambda \cdot \max(0,\ t_i^k - b_i)$$

où $\lambda$ est un paramètre de pondération à calibrer.

In [ ]:
# Visualisation des fenêtres temporelles sur un exemple de 4 clients
np.random.seed(GLOBAL_SEED)

n_viz = 4
horizon = 480  # journée de 8 heures en minutes (8h00 → 16h00)

# Génération de fenêtres temporelles cohérentes
ouvertures = np.random.randint(0, 300, size=n_viz)
largeurs    = np.random.randint(60, 150, size=n_viz)
fermetures  = np.minimum(ouvertures + largeurs, horizon)

# Heures d'arrivée simulées (pour illustrer les 3 cas)
arrivees = np.array([
    ouvertures[0] + 30,           # cas 1 : dans la fenêtre
    ouvertures[1] - 25,           # cas 2 : trop tôt
    fermetures[2] + 20,           # cas 3 : trop tard
    ouvertures[3] + largeurs[3] // 2  # cas 1 : dans la fenêtre
])

labels_clients = [f'Client {i+1}' for i in range(n_viz)]
couleurs_fenetres = '#B5D4F4'
couleur_ok      = '#1D9E75'
couleur_tot     = '#EF9F27'
couleur_tard    = '#E24B4A'

fig, ax = plt.subplots(figsize=(10, 4))

for i in range(n_viz):
    # Barre de la fenêtre temporelle
    ax.barh(i, fermetures[i] - ouvertures[i],
            left=ouvertures[i], height=0.4,
            color=couleurs_fenetres, edgecolor='#378ADD',
            linewidth=0.8, label='Fenêtre' if i == 0 else '')

    # Heure d'arrivée
    if arrivees[i] < ouvertures[i]:
        couleur = couleur_tot
        texte   = 'Trop tôt : attente'
    elif arrivees[i] > fermetures[i]:
        couleur = couleur_tard
        texte   = 'Trop tard : violation'
    else:
        couleur = couleur_ok
        texte   = 'Dans la fenêtre'

    ax.axvline(arrivees[i], ymin=(i) / n_viz + 0.05,
               ymax=(i + 1) / n_viz - 0.05,
               color=couleur, linewidth=2, linestyle='--')
    ax.text(arrivees[i] + 5, i, texte,
            va='center', fontsize=8.5, color=couleur)

# Étiquettes
ax.set_yticks(range(n_viz))
ax.set_yticklabels(labels_clients)
ax.set_xlabel('Temps (minutes depuis 8h00)')
ax.set_title('Fenêtres temporelles [aᵢ, bᵢ] et heures d\'arrivée simulées')

# Légende manuelle
from matplotlib.lines import Line2D
legende = [
    mpatches.Patch(color=couleurs_fenetres, label='Fenêtre [aᵢ, bᵢ]'),
    Line2D([0], [0], color=couleur_ok,   linestyle='--', label='Arrivée valide'),
    Line2D([0], [0], color=couleur_tot,  linestyle='--', label='Arrivée trop tôt'),
    Line2D([0], [0], color=couleur_tard, linestyle='--', label='Arrivée trop tard'),
]
ax.legend(handles=legende, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### 3.6 Contrainte C5 : Cohérence temporelle

$$t_i^k + s_i + c_{ij} \leq t_j^k \qquad \text{si } x_{ij}^k = 1,\
\forall (i,j) \in A,\ \forall k \in K$$

Si le véhicule $k$ va directement de $i$ à $j$, son heure d'arrivée en $j$
est au minimum égale à son heure d'arrivée en $i$, plus la durée de service
en $i$, plus le temps de trajet de $i$ vers $j$ :

$$t_j^k \geq t_i^k + s_i + c_{ij}$$

C'est la contrainte la plus structurante du VRPTW.
Elle propage le temps tout au long de la tournée :
un retard en début de tournée se répercute sur toutes les visites suivantes.

Combinée à C4, elle crée un couplage fort entre les décisions de routage
et le planning temporel. On ne peut pas choisir l'ordre des visites
uniquement selon les distances : il faut aussi vérifier
que les heures d'arrivée résultantes respectent les fenêtres.

Exemple numérique :

Soit $t_1^k = 60$ min, $s_1 = 10$ min, $c_{1,2} = 30$ min.

L'heure d'arrivée au client 2 est au minimum :
$t_2^k \geq 60 + 10 + 30 = 100$ min.

Si la fenêtre du client 2 est $[80, 120]$, l'arrivée à 100 min est valide.
Si la fenêtre est $[70, 95]$, l'arrivée à 100 min viole C4 :
il faudrait réordonner la tournée ou accepter la pénalité.

### 3.7 Modèle VRPTW complet

Nous disposons maintenant de tous les éléments du modèle.

Données :

- $V = \{0, 1, \ldots, n\}$ : sommets (0 = dépôt)
- $K = \{1, \ldots, m\}$ : véhicules
- $c_{ij} \in \mathbb{R}^+$ : coûts de trajet
- $q_i \in \mathbb{R}^+$ : demandes clients
- $Q \in \mathbb{R}^+$ : capacité des véhicules
- $[a_i, b_i]$ : fenêtres temporelles
- $s_i \in \mathbb{R}^+$ : durées de service

Variables de décision :

- $x_{ij}^k \in \{0, 1\}$ : arc $(i \to j)$ emprunté par le véhicule $k$
- $t_i^k \in \mathbb{R}^+$ : heure d'arrivée du véhicule $k$ au client $i$

Objectif et contraintes :

$$\min \sum_{k \in K} \sum_{i \in V} \sum_{j \in V} c_{ij} \cdot x_{ij}^k$$

$$\text{(C1)} \quad \sum_{k \in K} \sum_{j \in V} x_{ij}^k = 1
\qquad \forall i \in V \setminus \{0\}$$

$$\text{(C2)} \quad \sum_{i \in V} x_{ij}^k = \sum_{i \in V} x_{ji}^k
\qquad \forall j \in V,\ \forall k \in K$$

$$\text{(C3)} \quad \sum_{i \in V} q_i \cdot \left(\sum_{j \in V} x_{ij}^k\right) \leq Q
\qquad \forall k \in K$$

$$\text{(C4)} \quad a_i \leq t_i^k \leq b_i
\qquad \forall i \in V,\ \forall k \in K$$

$$\text{(C5)} \quad t_i^k + s_i + c_{ij} \leq t_j^k \quad \text{si } x_{ij}^k = 1
\qquad \forall (i,j) \in A,\ \forall k \in K$$

Ce modèle est un programme linéaire en nombres entiers mixte (MILP).
Sa résolution exacte est NP-difficile : ce que nous démontrons dans la section suivante.

---

## 4. Analyse de complexité

### 4.1 Classes de complexité

La complexité algorithmique mesure la quantité de ressources
(temps, mémoire) nécessaires à la résolution d'un problème
en fonction de la taille de l'entrée $n$.

La notation $O()$ exprime le comportement asymptotique :
on s'intéresse à ce qui se passe quand $n$ grandit,
indépendamment de la machine ou de l'implémentation.

Deux classes fondamentales structurent la théorie :

La classe $\mathbf{P}$ contient les problèmes résolubles en temps polynomial,
c'est-à-dire en $O(n^k)$ pour une constante $k$ fixée.
Ces problèmes sont considérés comme "efficacement résolubles".

La classe $\mathbf{NP}$ contient les problèmes dont une solution candidate
peut être vérifiée en temps polynomial, même si on ne sait pas
la trouver rapidement. Tout problème de $P$ est dans $NP$,
mais la réciproque est inconnue : c'est la question ouverte $P = NP ?$,
l'un des sept problèmes du millénaire de l'Institut Clay.

Un problème est dit $\mathbf{NP}$-difficile s'il est au moins aussi dur
que tous les problèmes de $NP$. Résoudre un problème NP-difficile
en temps polynomial résoudrait tous les problèmes de NP simultanément.
On conjecture que cela est impossible ($P \neq NP$).

Le TSP et le VRPTW appartiennent à cette dernière catégorie.

### 4.2 Explosion combinatoire

Une approche naïve pour résoudre le TSP consiste à énumérer
toutes les tournées possibles et à retenir la meilleure.

Pour $n$ clients, le nombre de tournées distinctes est :

$$\text{Tournées} = \frac{(n-1)!}{2}$$

On divise par $(n-1)$ car le point de départ est fixé (le dépôt),
et par $2$ car une tournée et son inverse ont le même coût
dans le cas euclidien.

La croissance factorielle est explosive :

| $n$ | Tournées | Temps estimé à $10^9$ op/s |
|-----|----------|---------------------------|
| 5   | 12 | $< 1$ ms |
| 10  | 181 440 | $< 1$ ms |
| 15  | 43 milliards | $\approx 43$ s |
| 20  | $6 \times 10^{16}$ | $\approx 2000$ ans |
| 50  | $\approx 10^{62}$ | $\gg$ âge de l'univers |


In [ ]:
# Calcul du nombre de tournées et comparaison des complexités
n_values = list(range(2, 16))

# Nombre de tournées exactes (force brute)
n_tournees = [math.factorial(n - 1) // 2 for n in n_values]

# Temps estimé en secondes à 10^9 opérations/seconde
ops_par_sec = 1e9
temps_sec   = [t / ops_par_sec for t in n_tournees]

# Complexités de référence pour comparaison
O_n2      = [n**2          for n in n_values]
O_2n      = [2**n          for n in n_values]
O_factoriel = [math.factorial(n - 1) // 2 for n in n_values]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Graphique 1 : comparaison des ordres de grandeur (échelle log)
axes[0].semilogy(n_values, O_n2,        label='O(n²)',   color='#1D9E75', linewidth=2)
axes[0].semilogy(n_values, O_2n,        label='O(2ⁿ)',   color='#EF9F27', linewidth=2)
axes[0].semilogy(n_values, O_factoriel, label='O((n-1)!/2)', color='#E24B4A', linewidth=2)
axes[0].set_xlabel('Nombre de villes n')
axes[0].set_ylabel('Nombre d\'opérations (échelle log)')
axes[0].set_title('Comparaison des complexités')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Graphique 2 : temps réel force brute
temps_annees = [t / (3600 * 24 * 365) for t in temps_sec]
axes[1].semilogy(n_values, temps_annees, color='#E24B4A', linewidth=2, marker='o', markersize=5)
axes[1].axhline(y=1,    color='#888780', linestyle='--', linewidth=1, label='1 an')
axes[1].axhline(y=13.8e9, color='#378ADD', linestyle='--', linewidth=1, label='Âge de l\'univers')
axes[1].set_xlabel('Nombre de villes n')
axes[1].set_ylabel('Temps de calcul (années, échelle log)')
axes[1].set_title('Temps force brute à 10⁹ op/s')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Explosion combinatoire du TSP : O((n-1)!/2)', y=1.02)
plt.tight_layout()
plt.show()

# Tableau synthétique
print(f"{'n':>4}  {'Tournées':>20}  {'Temps estimé':>20}")
print("-" * 50)
for n in [5, 10, 12, 15, 20]:
    t = math.factorial(n - 1) // 2
    s = t / ops_par_sec
    if s < 0.001:
        temps_str = "< 1 ms"
    elif s < 1:
        temps_str = f"{s*1000:.1f} ms"
    elif s < 3600:
        temps_str = f"{s:.1f} s"
    elif s < 86400 * 365:
        temps_str = f"{s/3600:.0f} heures"
    else:
        temps_str = f"{s/(86400*365):.2e} ans"
    print(f"{n:>4}  {t:>20,}  {temps_str:>20}")

### 4.3 Démonstration de NP-difficulté : réduction depuis le circuit hamiltonien

Pour prouver qu'un problème $B$ est NP-difficile, on utilise
une réduction polynomiale depuis un problème $A$ déjà connu comme NP-complet.
On note $A \leq_p B$ et on lit "A se réduit polynomialement à B".

Le raisonnement est le suivant : si on savait résoudre $B$ en temps polynomial,
alors on pourrait résoudre $A$ en temps polynomial aussi :
ce qui contredirait la conjecture $P \neq NP$.
Donc $B$ est au moins aussi dur que $A$, c'est-à-dire NP-difficile.

Nous réduisons le problème du circuit hamiltonien vers le TSP.

Le problème du circuit hamiltonien est le suivant :
étant donné un graphe quelconque $G = (V, E)$,
existe-t-il un cycle passant exactement une fois par chaque sommet ?
Ce problème est NP-complet : démontré par Karp (1972)
dans sa liste de 21 problèmes NP-complets.

La réduction se construit en quatre étapes :

Étape 1 : Construire une instance TSP depuis $G$.
On crée un graphe complet $G' = (V, K_n, w)$ sur les mêmes sommets,
avec des poids définis ainsi :

$$w(i,j) = \begin{cases} 0 & \text{si } (i,j) \in E \\ 1 & \text{sinon} \end{cases}$$

Cette construction prend un temps $O(n^2)$ : polynomial.

Étape 2 : Montrer l'équivalence.
$G$ possède un circuit hamiltonien
si et seulement si $G'$ admet une tournée TSP de coût total 0.

Si $G$ a un circuit hamiltonien, cette séquence de sommets forme
une tournée dans $G'$ n'utilisant que des arcs de poids 0,
donc de coût total 0.

Réciproquement, si $G'$ a une tournée de coût 0, elle n'utilise
que des arcs $(i,j)$ avec $w(i,j) = 0$, c'est-à-dire des arcs
qui existent dans $E$ : ce qui définit un circuit hamiltonien dans $G$.

Étape 3 : Conclure par l'absurde.
Supposons qu'il existe un algorithme polynomial pour le TSP.
On pourrait alors résoudre le circuit hamiltonien en temps polynomial :
construire $G'$ en $O(n^2)$, résoudre TSP en $O(n^k)$,
vérifier si le coût est 0 en $O(1)$.
Cela contredirait la NP-complétude du circuit hamiltonien.
Donc aucun algorithme polynomial pour le TSP n'existe (sauf si $P = NP$).

Étape 4 : Extension au VRPTW.
Le TSP est un cas particulier du VRPTW :
il suffit de poser $m = 1$ véhicule, $Q = \infty$ et $[a_i, b_i] = [0, \infty]$
pour obtenir exactement le TSP.
Donc TSP $\leq_p$ VRPTW, et le VRPTW est lui aussi NP-difficile.

In [ ]:
# Illustration de la réduction hamiltonien → TSP
# G : graphe quelconque (pas nécessairement complet)
# G': graphe complet avec poids 0/1

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Graphe G (hamiltonien)
G_hamilton = nx.Graph()
G_hamilton.add_nodes_from([0, 1, 2, 3, 4])
# Arêtes existantes (forment un circuit hamiltonien 0→1→2→3→4→0)
aretes_G = [(0,1), (1,2), (2,3), (3,4), (4,0), (0,2)]
G_hamilton.add_edges_from(aretes_G)

pos_h = nx.circular_layout(G_hamilton)

# Dessiner G
nx.draw_networkx_nodes(G_hamilton, pos_h, node_color='#B5D4F4',
                       node_size=600, ax=axes[0])
nx.draw_networkx_labels(G_hamilton, pos_h, font_size=10, ax=axes[0])
nx.draw_networkx_edges(G_hamilton, pos_h, ax=axes[0],
                       edge_color='#378ADD', width=1.5)
# Mettre en valeur le circuit hamiltonien
circuit = [(0,1), (1,2), (2,3), (3,4), (4,0)]
nx.draw_networkx_edges(G_hamilton, pos_h, edgelist=circuit,
                       ax=axes[0], edge_color='#E24B4A',
                       width=3, style='solid')
axes[0].set_title('G : graphe original\n(arêtes rouges = circuit hamiltonien)',
                  pad=10)
axes[0].axis('off')

# Graphe G' (TSP avec poids 0/1)
G_tsp = nx.complete_graph(5)
pos_t = nx.circular_layout(G_tsp)

# Séparer arêtes de poids 0 et poids 1
aretes_0 = [(i,j) for (i,j) in G_tsp.edges()
             if (i,j) in aretes_G or (j,i) in aretes_G]
aretes_1 = [(i,j) for (i,j) in G_tsp.edges()
             if (i,j) not in aretes_G and (j,i) not in aretes_G]

nx.draw_networkx_nodes(G_tsp, pos_t, node_color='#C0DD97',
                       node_size=600, ax=axes[1])
nx.draw_networkx_labels(G_tsp, pos_t, font_size=10, ax=axes[1])
nx.draw_networkx_edges(G_tsp, pos_t, edgelist=aretes_0,
                       ax=axes[1], edge_color='#1D9E75',
                       width=2.5, label='w = 0')
nx.draw_networkx_edges(G_tsp, pos_t, edgelist=aretes_1,
                       ax=axes[1], edge_color='#D3D1C7',
                       width=1, style='dashed', label='w = 1')

# Labels des poids
edge_labels_0 = {e: '0' for e in aretes_0}
edge_labels_1 = {e: '1' for e in aretes_1}
nx.draw_networkx_edge_labels(G_tsp, pos_t, edge_labels=edge_labels_0,
                              font_size=8, font_color='#0F6E56', ax=axes[1])
nx.draw_networkx_edge_labels(G_tsp, pos_t, edge_labels=edge_labels_1,
                              font_size=8, font_color='#888780', ax=axes[1])

axes[1].set_title("G' : graphe TSP complet\n(vert w=0 : arête dans G   |   gris w=1 : arête absente)",
                  pad=10)
axes[1].axis('off')

plt.suptitle('Réduction circuit hamiltonien → TSP', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print("Réduction vérifiée :")
print(f"  Arêtes dans G    : {aretes_G}")
print(f"  Arêtes de poids 0 dans G' : {aretes_0}")
print(f"  Le circuit 0→1→2→3→4→0 a un coût de : "
      f"{sum(1 if (circuit[i][0], circuit[i][1]) in aretes_1 else 0 for i in range(len(circuit)))} "
      f"dans G'  (attendu : 0)")

### 4.4 Conséquences pour le projet

La NP-difficulté du VRPTW a des conséquences directes sur les choix
algorithmiques du projet.

Il n'existe pas d'algorithme exact capable de résoudre une instance
de taille $n = 100$ en temps raisonnable : même avec les meilleurs
ordinateurs actuels. La force brute est inutilisable au-delà de $n = 15$.

Cela justifie l'architecture en trois méthodes que nous développerons :

Une méthode exacte par force brute, limitée à $n \leq 12$,
servira uniquement à valider les autres méthodes sur de petites instances.

Une méthode heuristique (NNH + 2-opt) fournira une solution approchée
rapidement, en $O(n^2)$. Elle servira de baseline qualité.

Une métaheuristique (recuit simulé) explorera intelligemment
l'espace des solutions pour s'approcher de l'optimal,
au prix d'un temps de calcul croissant avec $n$.

Une méthode par apprentissage par renforcement (Deep Learning)
apprendra une politique de construction de tournées par entraînement,
puis produira des solutions en temps d'inférence constant : indépendant de $n$.
C'est l'argument central de la comparaison expérimentale.

Aucune de ces méthodes ne garantit l'optimalité sur les grandes instances.
C'est une conséquence directe de la NP-difficulté, et non un défaut
de conception algorithmique.

---

## 5. Générateur d'instances

### 5.1 Choix et justification des paramètres

Le générateur produit des instances aléatoires du VRPTW
utilisées par toutes les phases de résolution du projet.
Sa conception doit garantir trois propriétés :

- Reproductibilité : deux exécutions avec la même seed produisent la même instance.
- Faisabilité : les instances générées admettent au moins une solution valide.
- Représentativité : les paramètres reflètent des cas réels de logistique urbaine.

Nous nous appuyons sur les instances de référence Solomon (1987),
le benchmark standard du VRPTW, pour calibrer nos paramètres.

Taille $n$ : nous générons des instances pour
$n \in \{10, 20, 50, 100, 200\}$ clients.
Ces tailles couvrent le spectre allant des petites instances
(résolubles exactement, pour validation) aux grandes instances
(inaccessibles à la force brute, pour l'étude expérimentale).

Coordonnées : les villes sont placées aléatoirement
dans un carré $[0, 100] \times [0, 100]$ avec une distribution uniforme.
Cela correspond aux instances de type R (Random) de Solomon.

Demandes $q_i$ : tirées uniformément dans $[1, 30]$ unités.
Cette plage garantit une variabilité réaliste des charges.

Capacité $Q$ : fixée à $0.3 \times \sum q_i / m$ en moyenne,
ce qui force l'utilisation de $m \approx 3$ véhicules.
Une capacité trop grande rendrait la contrainte inactive,
une capacité trop petite rendrait les instances infaisables.

Fenêtres temporelles $[a_i, b_i]$ : construites en deux étapes.
On calcule d'abord le temps de trajet depuis le dépôt vers chaque client.
La fenêtre est centrée autour de cette valeur avec une largeur
de $30\%$ de l'horizon total. Cela garantit la faisabilité
tout en maintenant une contrainte effective.

Durées de service $s_i$ : tirées uniformément dans $[5, 15]$ minutes,
représentant le temps de livraison sur place.

Horizon temporel : fixé à $T = 480$ minutes (journée de 8 heures).

In [ ]:
def generate_instance(n, n_vehicles=3, seed=42):
    """
    Génère une instance aléatoire du VRPTW.
    @param n : nombre de clients (hors dépôt)
    @param n_vehicles : nombre de véhicules disponibles
    @param seed : int, graine pour la reproductibilité
    @return : dict avec les données de l'instance
    """
    rng     = np.random.default_rng(seed)
    horizon = 480.0  # 8 heures en minutes

    # Facteur de conversion distance → minutes
    vitesse = 0.6  # minutes par unité de distance

    # Coordonnées
    depot   = np.array([[50.0, 50.0]])
    clients = rng.uniform(0, 100, size=(n, 2))
    coords  = np.concatenate([depot, clients], axis=0)

    # Matrice des distances euclidiennes
    diff  = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist  = np.sqrt((diff ** 2).sum(axis=2))

    # Matrice des durées de trajet en minutes
    durees = dist * vitesse

    # Demandes
    demands     = np.zeros(n + 1)
    demands[1:] = rng.integers(1, 31, size=n).astype(float)

    # Capacité
    total_demand = demands.sum()
    capacity     = total_demand / n_vehicles * 1.2

    # Durées de service
    service_times      = np.zeros(n + 1)
    service_times[1:]  = rng.uniform(5, 15, size=n)

    # Fenêtres temporelles — indépendantes des distances
    # Trois profils de clients simulant des comportements réels :
    #   - Strict  (40%) : créneau de 30 à 60 min, heure imposée
    #   - Modéré  (40%) : créneau de 90 à 150 min, demi-journée
    #   - Large   (20%) : créneau de 200 à 300 min, quasi disponible
    time_windows    = np.zeros((n + 1, 2))
    time_windows[0] = [0, horizon]  # dépôt : toute la journée

    profils = rng.choice(['strict', 'modere', 'large'],
                         size=n,
                         p=[0.4, 0.4, 0.2])

    for i in range(1, n + 1):
        profil = profils[i - 1]

        if profil == 'strict':
            # Créneau court : 30 à 60 min
            largeur = rng.uniform(30, 60)
            # Heure d'ouverture répartie sur toute la journée
            # en laissant de la place pour la durée du créneau
            a_i = rng.uniform(0, horizon - largeur)

        elif profil == 'modere':
            # Créneau moyen : 90 à 150 min
            largeur = rng.uniform(90, 150)
            a_i     = rng.uniform(0, horizon - largeur)

        else:  # large
            # Créneau large : 200 à 300 min
            largeur = rng.uniform(200, 300)
            a_i     = rng.uniform(0, horizon - largeur)

        b_i = min(horizon, a_i + largeur)
        time_windows[i] = [round(a_i, 1), round(b_i, 1)]

    return {
        'n'             : n,
        'n_vehicles'    : n_vehicles,
        'coords'        : coords,
        'dist'          : dist,
        'durees'        : durees,
        'demands'       : demands,
        'capacity'      : round(capacity, 1),
        'time_windows'  : time_windows,
        'service_times' : service_times,
        'horizon'       : horizon,
        'vitesse'       : vitesse,
        'seed'          : seed,
    }

In [ ]:
def visualiser_instance(instance):
    """
    Affiche une instance VRPTW : carte des villes et fenêtres temporelles.
    @param instance : dict avec les données de l'instance
    @return : None (affiche les graphiques)
    """
    coords       = instance['coords']
    time_windows = instance['time_windows']
    demands      = instance['demands']
    n            = instance['n']
    horizon      = instance['horizon']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Graphique 1 : carte des villes
    ax = axes[0]
    ax.scatter(coords[1:, 0], coords[1:, 1],
               c='#378ADD', s=80, zorder=3, label='Clients')
    ax.scatter(coords[0, 0], coords[0, 1],
               c='#E24B4A', s=160, marker='*', zorder=4, label='Dépôt')

    # Numéroter les clients
    for i in range(1, n + 1):
        ax.annotate(str(i),
                    xy=(coords[i, 0], coords[i, 1]),
                    xytext=(4, 4), textcoords='offset points',
                    fontsize=7.5, color='#0C447C')

    ax.set_title(f'Carte des villes — n={n} clients')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=9)
    ax.set_xlim(-5, 105)
    ax.set_ylim(-5, 105)

    # Graphique 2 : fenêtres temporelles
    ax2 = axes[1]

    # Afficher les 10 premiers clients pour lisibilité
    n_affiche = min(n, 10)
    for i in range(1, n_affiche + 1):
        a_i, b_i = time_windows[i]
        ax2.barh(i, b_i - a_i, left=a_i,
                 height=0.5, color='#B5D4F4',
                 edgecolor='#378ADD', linewidth=0.8)
        ax2.text(b_i + 3, i,
                 f'q={int(demands[i])}',
                 va='center', fontsize=8, color='#5F5E5A')

    ax2.set_yticks(range(1, n_affiche + 1))
    ax2.set_yticklabels([f'Client {i}' for i in range(1, n_affiche + 1)],
                        fontsize=8.5)
    ax2.set_xlabel('Temps (minutes depuis 8h00)')
    ax2.set_xlim(0, horizon + 40)
    ax2.set_title(f'Fenêtres temporelles (10 premiers clients)\n'
                  f'Capacité Q = {instance["capacity"]:.0f} unités')
    ax2.axvline(horizon, color='#E24B4A', linewidth=1,
                linestyle='--', label=f'Horizon ({int(horizon)} min)')
    ax2.legend(fontsize=9)

    plt.suptitle(f'Instance VRPTW — n={n}, '
                 f'{instance["n_vehicles"]} véhicules, seed={instance["seed"]}',
                 fontsize=12)
    plt.tight_layout()
    plt.show()


# Démonstration sur n=10
instance_demo = generate_instance(n=10, n_vehicles=3, seed=GLOBAL_SEED)
visualiser_instance(instance_demo)

In [ ]:
def verifier_faisabilite(instance):
    """
    Vérifie qu'une instance VRPTW est théoriquement faisable.
    @param instance : dict avec les données de l'instance
    @return : dict avec les résultats des vérifications
    """
    dist         = instance['dist']
    time_windows = instance['time_windows']
    demands      = instance['demands']
    capacity     = instance['capacity']
    n_vehicles   = instance['n_vehicles']
    n            = instance['n']

    resultats = {
        'faisable'          : True,
        'problemes'         : [],
        'clients_inatteignables' : [],
        'couverture_demande': True,
    }

    # Contrôle 1 : atteignabilité depuis le dépôt
    for i in range(1, n + 1):
        t_arrivee_min = dist[0, i]  # temps minimal de trajet depuis dépôt
        a_i, b_i      = time_windows[i]
        if t_arrivee_min > b_i:
            resultats['faisable'] = False
            resultats['clients_inatteignables'].append(i)
            resultats['problemes'].append(
                f'Client {i} inatteignable : trajet min={t_arrivee_min:.1f} '
                f'> fermeture b_i={b_i:.1f}'
            )

    # Contrôle 2 : couverture de la demande totale
    demande_totale    = demands[1:].sum()
    capacite_totale   = capacity * n_vehicles
    if capacite_totale < demande_totale:
        resultats['faisable']           = False
        resultats['couverture_demande'] = False
        resultats['problemes'].append(
            f'Capacité totale {capacite_totale:.1f} '
            f'< demande totale {demande_totale:.1f}'
        )

    # Contrôle 3 : validité des fenêtres
    for i in range(1, n + 1):
        a_i, b_i = time_windows[i]
        if a_i >= b_i:
            resultats['faisable'] = False
            resultats['problemes'].append(
                f'Client {i} : fenêtre invalide a_i={a_i} >= b_i={b_i}'
            )

    return resultats


# Vérification sur plusieurs tailles
print(f"{'n':>6}  {'Faisable':>10}  {'Inatteignables':>16}  "
      f"{'Demande':>10}  {'Capacité':>10}")
print("-" * 60)

for n_test in [10, 20, 50, 100, 200]:
    inst = generate_instance(n=n_test, n_vehicles=3, seed=GLOBAL_SEED)
    res  = verifier_faisabilite(inst)
    print(
        f"{n_test:>6}  "
        f"{'Oui' if res['faisable'] else 'Non':>10}  "
        f"{len(res['clients_inatteignables']):>16}  "
        f"{inst['demands'][1:].sum():>10.1f}  "
        f"{inst['capacity'] * inst['n_vehicles']:>10.1f}"
    )

In [ ]:
def sauvegarder_instance(instance, chemin):
    """
    @param instance : dict avec les données de l'instance
    @param chemin : str, chemin du fichier de sortie (ex: 'instance.json')
    @return : None (sauvegarde l'instance au format JSON)
    Note : les tableaux numpy sont convertis en listes pour être JSON-compatibles.
    """
    instance_serialisable = {
        k: v.tolist() if isinstance(v, np.ndarray) else v
        for k, v in instance.items()
    }
    with open(chemin, 'w', encoding='utf-8') as f:
        json.dump(instance_serialisable, f, indent=2, ensure_ascii=False)


def resumer_instance(instance):
    """
    Affiche un résumé lisible d'une instance VRPTW.
    """
    n   = instance['n']
    tw  = instance['time_windows']
    dem = instance['demands']

    print(f"Instance VRPTW — seed {instance['seed']}")
    print(f"  Clients       : {n}")
    print(f"  Véhicules     : {instance['n_vehicles']}")
    print(f"  Capacité Q    : {instance['capacity']:.1f} unités")
    print(f"  Demande tot.  : {dem[1:].sum():.1f} unités")
    print(f"  Horizon       : {instance['horizon']:.0f} min")
    print(f"  TW min. larg. : {(tw[1:,1] - tw[1:,0]).min():.1f} min")
    print(f"  TW max. larg. : {(tw[1:,1] - tw[1:,0]).max():.1f} min")
    print(f"  TW moy. larg. : {(tw[1:,1] - tw[1:,0]).mean():.1f} min")
    print(f"  Dist. moy.    : {instance['dist'][instance['dist'] > 0].mean():.1f}")


# Démonstration complète
instance_demo = generate_instance(n=10, n_vehicles=3, seed=GLOBAL_SEED)
resumer_instance(instance_demo)

# Sauvegarde
sauvegarder_instance(instance_demo, 'instance_n10_seed42.json')
print("\nInstance sauvegardée : instance_n10_seed42.json")

# Vérification du rechargement
with open('instance_n10_seed42.json', 'r') as f:
    instance_rechargee = json.load(f)

print(f"Rechargement OK — n = {instance_rechargee['n']}, "
      f"seed = {instance_rechargee['seed']}")